In [2]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

In [3]:
import torch,random
import tensorflow as tf
import numpy as np
def random_seed(seed=1727, use_cuda=False):
    np.random.seed(seed) # cpu vars
    torch.manual_seed(seed) # cpu vars
    random.seed(seed) # Python
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    if use_cuda: 
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
        
random_seed(1727)
tf.config.list_physical_devices()

[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]

In [4]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [5]:
compName = Path('nlp-getting-started')

In [6]:
if not iskaggle:
    if not compName.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(compName))
        zipfile.ZipFile(f'{compName}.zip').extractall(compName)
else:
    # /kaggle/input/competitions/nlp-getting-started/train.csv
    compName = Path(f'/kaggle/input/competitions/{compName}')

# %pip install -q datasets
!dir /o:g /w {compName}
# !ls {compName}

 Volume in drive C is Windows
 Volume Serial Number is 6291-898F

 Directory of c:\Users\longnuub\learning-programming-languages\learning-python\kaggle\nlp-getting-started

[..]                    [.]                     test.csv
train.csv               sample_submission.csv   
               3 File(s)      1.431.241 bytes
               2 Dir(s)  132.294.946.816 bytes free


In [7]:
import pandas as pd
train=pd.read_csv(compName/"train.csv")
test=pd.read_csv(compName/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["id"],inplace=True)

# feat engineering

In [8]:
train["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in train["keyword"]
])
train["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in train["location"]
])
train["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else 0 for keyword,text in zip(train["keyword"],train["text"])
])
train["has_all_caps"] = (
    train["text"].str.contains(r'\b[A-Z]{3,}\b')
).astype(int)

In [9]:
import re 

def clean_text(text):
    """Clean and preprocess text without NLTK"""
    if pd.isna(text):
        return ""
    
    text = text.lower() # Convert to lowercase
    text = re.sub(r'http\S+|www\S+|https\S+', '', text) # Remove URLs
    text = re.sub(r'@(\w+)', r'\1', text) # Remove mentions (keep the word without @ or #)
    text = re.sub(r'#(\w+)', r'\1', text) # Remove hashtags (keep the word without @ or #)
    text = re.sub(r'\d+', '', text) # Remove numbers
    text = ' '.join(text.split()) # Remove extra whitespace
    
    return text

In [10]:
# Basic text features
train["text_length"] = train["text"].str.len().fillna(0)
train["word_count"] = train["text"].str.split().str.len().fillna(0)

# Additional features (no external libraries)
train["capital_count"] = train["text"].str.findall(r'[A-Z]').str.len()
train["exclamation_count"] = train["text"].str.count('!')
train["question_count"] = train["text"].str.count('\\?')
train["hashtag_count"] = train["text"].str.count('#')
train["mention_count"] = train["text"].str.count('@')

# apply preproc
train["clean_text"] = train["text"].apply(clean_text)

In [11]:
train

,keyword,location,text,target,keywordDefined,locationDefined,keywordInText,has_all_caps,text_length,word_count,capital_count,exclamation_count,question_count,hashtag_count,mention_count,clean_text
0,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1,0,0,0,1,69,13,10,0,0,1,0,our deeds are the reason of this earthquake ma...
1,NaN,NaN,Forest fire near La Ronge Sask. Canada,1,0,0,0,0,38,7,5,0,0,0,0,forest fire near la ronge sask. canada
2,NaN,NaN,All residents asked to 'shelter in place' are ...,1,0,0,0,0,133,22,2,0,0,0,0,all residents asked to 'shelter in place' are ...
3,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1,0,0,0,0,65,8,1,0,0,1,0,", people receive wildfires evacuation orders i..."
4,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1,0,0,0,0,88,16,3,0,0,2,0,just got sent this photo from ruby alaska as s...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7608,NaN,NaN,Two giant cranes holding a bridge collapse int...,1,0,0,0,0,83,11,7,0,0,0,0,two giant cranes holding a bridge collapse int...
7609,NaN,NaN,@aria_ahrary @TheTawniest The out of control w...,1,0,0,0,0,125,20,6,0,0,0,2,aria_ahrary thetawniest the out of control wil...
7610,NaN,NaN,M1.94 [01:04 UTC]?5km S of Volcano Hawaii. htt...,1,0,0,0,1,65,8,10,0,1,0,0,m. [: utc]?km s of volcano hawaii.
7611,NaN,NaN,Police investigating after an e-bike collided ...,1,0,0,0,0,137,19,4,0,0,0,0,police investigating after an e-bike collided ...


# prep

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

# Prepare text features
word_tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
char_tfidf = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3,5),
    max_features=10000,
    sublinear_tf=True
)
X_train_word = word_tfidf.fit_transform(train["clean_text"])
X_train_char = char_tfidf.fit_transform(train["clean_text"])

# Prepare numeric features
numeric_features = train[[
    "text_length", "word_count", "capital_count", 
    "exclamation_count", "question_count", 
    "hashtag_count", "mention_count", "keywordInText",
    "keywordDefined", "locationDefined", "has_all_caps",
]].fillna(0).values

scaler = StandardScaler()
numeric_features = scaler.fit_transform(numeric_features)

# Combine features
X = sp.hstack([X_train_word, X_train_char, numeric_features])
y = train["target"].values

In [13]:
from sklearn.model_selection import train_test_split, cross_val_score

xtrain,xval,ytrain,yval = train_test_split(
    X, y, test_size=0.2, random_state=RANDOMSEED, stratify=y
)

# training

we'll be ensemblemaxxing for this problem

In [14]:
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import GridSearchCV

print("grid search on linear svc:")
gslsvc=GridSearchCV(
  estimator=CalibratedClassifierCV(
    LinearSVC(max_iter=35000,random_state=RANDOMSEED),
    n_jobs=-1,
  ),
  param_grid={
    "estimator__C":[0.1, 0.25, 0.5, 0.75, 0.8, 0.95, 1, 1.5],
  },
  scoring=make_scorer(f1_score),
  verbose=3,
  n_jobs=-1,
)
gslsvc.fit(xtrain,ytrain)
print(f"Best gslsvc params: {gslsvc.best_params_}\nbest gslsvc f1: {gslsvc.best_score_}")

print("\ngrid search on logistic regression:")
gslr=GridSearchCV(
	estimator=LogisticRegression(max_iter=35000, random_state=RANDOMSEED),
	param_grid={
		"C": [0.1, 0.25, 0.5, 0.75, 0.8, 0.95, 1, 1.5],
		"solver": ["liblinear", "lbfgs", "saga"],
		"class_weight": [None, "balanced"],
	},
	scoring=make_scorer(f1_score),
  	verbose=3,
    n_jobs=-1,
)
gslr.fit(xtrain,ytrain)
print(f"Best gslr params: {gslr.best_params_}\nbest gslr f1: {gslr.best_score_}")

grid search on linear svc:
Fitting 5 folds for each of 8 candidates, totalling 40 fits
Best gslsvc params: {'estimator__C': 0.25}
best gslsvc f1: 0.7525399710111793

grid search on logistic regression:
Fitting 5 folds for each of 48 candidates, totalling 240 fits
Best gslr params: {'C': 1.5, 'class_weight': 'balanced', 'solver': 'saga'}
best gslr f1: 0.7537214001292912


In [25]:
model = VotingClassifier(
    estimators=[
        ('lr', gslr.best_estimator_),
        ('lsvc', gslsvc.best_estimator_),
    ],
    voting='soft' 
)

model.fit(xtrain,ytrain)

,"estimators estimators: list of (str, estimator) tuplesInvoking the ``fit`` method on the ``VotingClassifier`` will fit clonesof those original estimators that will be stored in the class attribute``self.estimators_``. An estimator can be set to ``'drop'`` using:meth:`set_params`... versionchanged:: 0.21 ``'drop'`` is accepted. Using None was deprecated in 0.22 and support was removed in 0.24.","[('lr', ...), ('lsvc', ...)]"
,"voting voting: {'hard', 'soft'}, default='hard'If 'hard', uses predicted class labels for majority rule voting.Else if 'soft', predicts the class label based on the argmax ofthe sums of the predicted probabilities, which is recommended foran ensemble of well-calibrated classifiers.",'soft'
,"weights weights: array-like of shape (n_classifiers,), default=NoneSequence of weights (`float` or `int`) to weight the occurrences ofpredicted class labels (`hard` voting) or class probabilitiesbefore averaging (`soft` voting). Uses uniform weights if `None`.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel for ``fit``.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionadded:: 0.18",None
,"flatten_transform flatten_transform: bool, default=TrueAffects shape of transform output only when voting='soft'If voting='soft' and flatten_transform=True, transform method returnsmatrix with shape (n_samples, n_classifiers * n_classes). Ifflatten_transform=False, it returns(n_classifiers, n_samples, n_classes).",True
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting will be printed as itis completed... versionadded:: 0.23",False
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.5
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001


In [26]:
probs = model.predict_proba(xval)[:,1]
best_f1 = 0
best_thresh = 0.5

for t in np.arange(0.1, 0.9, 0.01):
    preds = (probs > t).astype(int)
    f1 = f1_score(yval, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print(f"best threshold: {best_thresh}\nbest f1: {best_f1}")

best threshold: 0.45999999999999985
best f1: 0.79879427279578


In [ ]:
model.fit(X,y) # fit on full X & y 

# predictions

feat engineering for test data

In [28]:
test["keywordDefined"]=pd.Series([
  int(pd.notna(keyword)) for keyword in test["keyword"]
])
test["locationDefined"]=pd.Series([
  int(pd.notna(loc)) for loc in test["location"]
])
test["keywordInText"]=pd.Series([
	int(str(keyword).lower() in str(text).lower()) if keyword else 0 for keyword,text in zip(test["keyword"],test["text"])
])
test["has_all_caps"] = (
    test["text"].str.contains(r'\b[A-Z]{3,}\b')
).astype(int)

test["clean_text"] = test["text"].apply(clean_text)

test["text_length"] = test["text"].str.len().fillna(0)
test["word_count"] = test["text"].str.split().str.len().fillna(0)
test["capital_count"] = test["text"].str.findall(r'[A-Z]').str.len()
test["exclamation_count"] = test["text"].str.count('!')
test["question_count"] = test["text"].str.count('\\?')
test["hashtag_count"] = test["text"].str.count('#')
test["mention_count"] = test["text"].str.count('@')

prep

In [29]:
X_test_word = word_tfidf.transform(test["clean_text"])
X_test_char = char_tfidf.transform(test["clean_text"])

test_numeric = test[[
    "text_length", "word_count", "capital_count", 
    "exclamation_count", "question_count", 
    "hashtag_count", "mention_count", "keywordInText",
    "keywordDefined", "locationDefined", "has_all_caps",
]].fillna(0).values
test_numeric = scaler.transform(test_numeric)

xtest = sp.hstack([X_test_word, X_test_char, test_numeric])

predict

In [30]:
preds = model.predict_proba(xtest)[:,1]
preds = (preds > best_thresh).astype(int)

In [31]:
print(f"Unique prediction values: {np.unique(preds)}")

Unique prediction values: [0 1]


In [32]:
subs_df=pd.DataFrame({
  "ID":test["id"],
  "Target":preds,
})

In [33]:
# subs_df.head()
subs_df.shape

(3263, 2)

In [34]:
subs_df.to_csv("submission.csv",index=False)